# Compute mutation rates for subset trees

Read mutation counts from host-specific, geographic, and temporal subtrees and compute basic rates.

In [ ]:
import os
import glob
import yaml
import pandas as pd
import numpy as np

In [ ]:
# Load config
with open('../config.yaml') as f:
    config = yaml.safe_load(f)

segments = config['segments']
ha_subtypes = config['ha_subtypes']
na_subtypes = config['na_subtypes']

# Define subset groups and their types
subset_groups = {
    'host': config['host_groups'],
    'geographic': config['geographic_groups'],
    'temporal': config['temporal_groups'],
}
print('Subset groups:', subset_groups)

In [ ]:
def get_subtypes_for_segment(segment):
    if segment == 'HA':
        return ha_subtypes
    elif segment == 'NA':
        return na_subtypes
    else:
        return ['all']


def read_counts(f, segment, subtype, subset, subset_type):
    """Read a mutation_counts.csv file and add metadata columns."""
    df = pd.read_csv(f, keep_default_na=False, low_memory=False)
    df = df.replace('', np.nan)
    df['subtype'] = subtype
    df['segment'] = segment
    df['segment_subtype'] = segment + '_' + subtype
    df['segment_length'] = df['site'].max() - df['site'].min()
    df['subset'] = subset
    df['subset_type'] = subset_type
    return df

In [ ]:
# Read mutation counts from all subset trees
dfs = []
for segment in segments:
    for subtype in get_subtypes_for_segment(segment):
        for subset_type, groups in subset_groups.items():
            for group in groups:
                f = f'../results/{segment}/{subtype}/{group}/mutation_counts.csv'
                if not os.path.exists(f):
                    print(f'WARNING: missing {f}')
                    continue
                df = read_counts(f, segment, subtype, group, subset_type)
                dfs.append(df)

counts_df = (
    pd.concat(dfs)
    .rename(columns={'parent_motif': 'motif'})
)

# Compute evolutionary opportunities and rates
counts_df['evo_opp'] = counts_df['syn_branch_length'] / counts_df['segment_length']
counts_df['rate'] = counts_df['actual_count'] / counts_df['evo_opp']

# Remove rows with incomplete motifs
counts_df = counts_df[
    (counts_df['motif'].notnull()) &
    (counts_df['motif'].str.len() == 3)
]

# Save
counts_df.sort_values('mut_type', inplace=True)
counts_df.to_csv('../results/subset_counts.csv', index=False)

print(f'Total rows: {len(counts_df):,}')
print(f'Subsets: {counts_df["subset"].unique()}')
counts_df.head()